# M06-P01 — RAG From Scratch

En M05 aprendimos a medir contexto y a recuperar documentos por similitud semántica. Ahora construimos el pipeline RAG completo sin frameworks para poder inspeccionar cada paso.

```text
documentos → chunks → embeddings → retrieval → contexto → prompt → LLM → respuesta con fuentes
```

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from enterprise_genai.config import get_settings
from enterprise_genai.embeddings import DEFAULT_DIMENSIONS
from enterprise_genai.rag import (
    answer_with_context,
    build_context,
    build_rag_index,
    chunk_documents,
    retrieve_chunks,
)

settings = get_settings()
CORPUS_PATH = Path('data/knowledge/m05_enterprise_chunks.json')
OUT_DIR = Path('outputs/m06_p01')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Modelo de embeddings:', settings.bedrock_embedding_model_id)
print('Modelo generativo:', settings.bedrock_text_model_id)
print('Dimensiones:', DEFAULT_DIMENSIONS)

## 1. Loaders y chunking

Reutilizamos los 80 documentos empresariales de M05. El primer paso del RAG es convertir cada documento en unidades recuperables. Usamos un chunker sencillo por palabras para que la mecánica sea visible.

El solapamiento evita que una idea situada justo en el borde desaparezca completamente de uno de los chunks.

In [ ]:
documents = json.loads(CORPUS_PATH.read_text(encoding='utf-8'))
CHUNK_SIZE = 36
OVERLAP = 8
chunks = chunk_documents(documents, chunk_size_words=CHUNK_SIZE, overlap_words=OVERLAP)

print('Documentos fuente:', len(documents))
print('Chunks generados:', len(chunks))
print('Chunk size:', CHUNK_SIZE, 'palabras')
print('Overlap:', OVERLAP, 'palabras')

chunk_df = pd.DataFrame(chunks)
display(chunk_df[['id', 'document_id', 'service', 'source_type', 'chunk_no', 'start_word', 'end_word']].head(12))

sample_doc_id = chunks[0]['document_id']
print('\nDocumento de ejemplo:', sample_doc_id)
for chunk in [c for c in chunks if c['document_id'] == sample_doc_id]:
    print(f"\n{chunk['id']} [{chunk['start_word']}:{chunk['end_word']}]\n{chunk['content']}")

## 2. Embeddings e índice vectorial

Cada chunk se transforma en un embedding normalizado de 512 dimensiones. La matriz resultante es nuestro índice vectorial mínimo en memoria. Todavía no usamos una base vectorial.

In [ ]:
started = time.perf_counter()
rag_index = build_rag_index(chunks, dimensions=DEFAULT_DIMENSIONS, max_workers=8)
index_elapsed_ms = round((time.perf_counter() - started) * 1000)
matrix = rag_index['matrix']

print('Forma de la matriz:', matrix.shape)
print('Llamadas de embedding:', rag_index['request_count'])
print('Tiempo de construcción del índice:', index_elapsed_ms, 'ms')
print('Norma primer vector:', round(float(np.linalg.norm(matrix[0])), 6))
print('Tokens totales embebidos:', sum(x or 0 for x in rag_index['token_counts']))

## 3. Retrieval

La consulta se convierte también en embedding. Después calculamos similitud coseno contra todos los chunks y conservamos los `top-k` más próximos.

In [ ]:
QUERY = '¿Qué debo comprobar si no recibo el segundo factor al conectarme por VPN?'
retrieval = retrieve_chunks(QUERY, chunks, matrix, top_k=4)

rows = []
for item in retrieval['results']:
    chunk = item['document']
    rows.append({
        'rank': item['rank'],
        'score': round(item['score'], 4),
        'chunk_id': chunk['id'],
        'document_id': chunk['document_id'],
        'service': chunk['service'],
        'title': chunk['title'],
    })

print('Consulta:', QUERY)
print('Tokens embedding consulta:', retrieval['query_tokens'])
print('Latencia embedding consulta:', retrieval['query_latency_ms'], 'ms')
display(pd.DataFrame(rows))

## 4. Construimos el contexto que verá el LLM

Retrieval todavía no responde la pregunta. Ahora convertimos los chunks recuperados en un bloque de contexto explícito. Éste es exactamente el texto que posteriormente enviaremos al modelo.

In [ ]:
context = build_context(retrieval['results'])
print(context)
print('\nCaracteres de contexto:', len(context))

## 5. Generation: respuesta fundamentada

El LLM recibe la pregunta y únicamente el contexto recuperado. El system prompt exige citar `[S1]`, `[S2]`, etc. y reconocer cuando el contexto no sea suficiente.

In [ ]:
rag_answer = answer_with_context(QUERY, retrieval['results'])
print(rag_answer['answer'])
print('\nUso:', rag_answer['usage'])
print('Latencia Bedrock:', rag_answer['metrics'].get('latencyMs'), 'ms')
print('Stop reason:', rag_answer['stop_reason'])

## 6. Abrir la caja: prompt completo

Una librería RAG suele construir este prompt por nosotros. Aquí lo mostramos para distinguir claramente tres elementos: pregunta, contexto recuperado e instrucciones de grounding.

In [ ]:
print('--- SYSTEM PROMPT ---')
print(rag_answer['system_prompt'])
print('\n--- USER PROMPT ---')
print(rag_answer['user_prompt'])

## 7. Persistimos el índice didáctico

En producción usaríamos un vector store. Aquí guardamos matriz y chunks para hacer tangible qué artefactos produce la fase de ingesta.

In [ ]:
np.savez_compressed(OUT_DIR / 'rag_embeddings.npz', embeddings=matrix)
(OUT_DIR / 'rag_chunks.json').write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding='utf-8')

print('Embeddings:', OUT_DIR / 'rag_embeddings.npz')
print('Chunks:', OUT_DIR / 'rag_chunks.json')
print('Siguiente paso: probar el modo RAG de Knowledge en Streamlit.')

## Qué acabamos de construir

```text
80 documentos
    ↓
chunking + overlap
    ↓
embeddings Titan
    ↓
similitud coseno / top-k
    ↓
contexto explícito
    ↓
Claude Haiku 4.5
    ↓
respuesta fundamentada + fuentes
```

En M06-P02 construiremos el mismo patrón con LangChain y mediremos la calidad del retrieval.